In [1]:
from google.colab import userdata, drive
from huggingface_hub import login
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer

In [2]:
drive.mount('drive')

Mounted at drive


In [3]:
HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)

In [4]:
dataset = load_dataset("azizdevlab/uzbek_corpus")
print(dataset)

README.md:   0%|          | 0.00/321 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/220M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/92.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3011581 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 3011581
    })
})


In [5]:
tokenizer = AutoTokenizer.from_pretrained("azizdevlab/gpt2-small-uzbek")

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
input_ids_list = []
attention_mask_list = []
def tokenizer_fn(example):
  enc = tokenizer(text=example['text'])
  input_ids_list.extend(enc['input_ids'])
  attention_mask_list.extend(enc['attention_mask'])

In [8]:
indices = list(range(1505791, 2323710))
chunk1 = dataset['train'].select(indices)

In [9]:
chunk1.map(tokenizer_fn, remove_columns=['text'])

Map:   0%|          | 0/817919 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 817919
})

In [10]:
def chunk_generator(input_ids, attention_mask, chunk_size):
    for i in range(0, len(input_ids), chunk_size):
        chunk_input = input_ids[i:i+chunk_size]
        chunk_attn = attention_mask[i:i+chunk_size]
        if len(chunk_input) == chunk_size:
            yield {"input_ids": chunk_input, "attention_mask": chunk_attn}

chunk_size = 768

dataset_chunked = Dataset.from_generator(
    lambda: chunk_generator(input_ids_list, attention_mask_list, chunk_size)
)

Generating train split: 0 examples [00:00, ? examples/s]

In [11]:
print(dataset_chunked)

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 193979
})


In [12]:
dataset_chunked.save_to_disk('/content/drive/MyDrive/uzbek_corpus/dataset_chunked3')

Saving the dataset (0/2 shards):   0%|          | 0/193979 [00:00<?, ? examples/s]

In [13]:
dataset_chunked=0
dataset1 = 0

In [14]:
input_ids_list = []
attention_mask_list = []
print(dataset_chunked)

0


In [15]:
indices = list(range(2323711, 3011581))
chunk = dataset['train'].select(indices)

In [16]:
tokenized_chunk = chunk.map(tokenizer_fn, remove_columns=['text'])

Map:   0%|          | 0/687870 [00:00<?, ? examples/s]

In [17]:
def chunk_generator(input_ids, attention_mask, chunk_size):
    for i in range(0, len(input_ids), chunk_size):
        chunk_input = input_ids[i:i+chunk_size]
        chunk_attn = attention_mask[i:i+chunk_size]
        if len(chunk_input) == chunk_size:
            yield {"input_ids": chunk_input, "attention_mask": chunk_attn}

chunk_size = 768

dataset_chunked2 = Dataset.from_generator(
    lambda: chunk_generator(input_ids_list, attention_mask_list, chunk_size)
)

Generating train split: 0 examples [00:00, ? examples/s]

In [18]:
print(dataset_chunked2)

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 13687
})


In [19]:
dataset_chunked2.save_to_disk('/content/drive/MyDrive/uzbek_corpus/dataset_chunked4')

Saving the dataset (0/1 shards):   0%|          | 0/13687 [00:00<?, ? examples/s]

In [1]:
from google.colab import drive
from datasets import load_from_disk, concatenate_datasets

In [2]:
drive.mount('drive')

Mounted at drive


In [6]:
dataset1 = load_from_disk("/content/drive/MyDrive/uzbek_corpus/dataset_chunked1")
dataset2 = load_from_disk("/content/drive/MyDrive/uzbek_corpus/dataset_chunked2")
dataset3 = load_from_disk("/content/drive/MyDrive/uzbek_corpus/dataset_chunked3")
dataset4 = load_from_disk("/content/drive/MyDrive/uzbek_corpus/dataset_chunked4")

In [7]:
print(dataset1)
print(dataset2)
print(dataset3)
print(dataset4)

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 264068
})
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 220737
})
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 193979
})
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 13687
})


In [9]:
dataset_combined = concatenate_datasets([dataset1, dataset2, dataset3, dataset4])
print(dataset_combined)

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 692471
})


In [10]:
dataset_combined.save_to_disk('/content/drive/MyDrive/uzbek_corpus/dataset_combined')

Saving the dataset (0/6 shards):   0%|          | 0/692471 [00:00<?, ? examples/s]